# 07 — Multi-Backbone Binary Classification

**Goal:** Train 4 additional transformer backbones on all 3 binary datasets to build a comprehensive backbone comparison table.

**Backbones (new):**
1. `xlm-roberta-base` — Strong multilingual model
2. `bert-base-multilingual-cased` (mBERT) — Classic multilingual baseline
3. `sagorsarker/bangla-bert-base` — Alternative Bengali-specific BERT
4. `google/muril-base-cased` (MuRIL) — Multilingual Representations for Indian Languages

**Existing backbone (from notebook 05):** `csebuetnlp/banglabert` — results loaded from saved CSVs

**Datasets:** Ben-Sarc binary, BanglaSarc binary, BanglaSarc3 binary

**Protocol:** Same hyperparameters as notebook 05 for fair comparison:  
epochs=3, lr=2e-5, batch_size=8, max_length=128, metric=macro_f1

## 0. Setup & Paths

In [14]:
import os
from pathlib import Path

def find_project_root():
    # If running on Kaggle
    if os.path.exists('/kaggle/working'):
        base = Path('/kaggle/working')
    else:
        base = Path.cwd()

    # Walk up until we find marker folders
    current = base
    while current != current.parent:
        if (current / '01_data').exists() and (current / '02_notebooks').exists():
            return current
        current = current.parent

    return base  # fallback

PROJECT = find_project_root()

SPLIT_DIR   = PROJECT / '01_data' / 'interim' / 'splits'
CKPT_DIR    = PROJECT / '03_models' / 'checkpoints'
TABLE_DIR   = PROJECT / '04_outputs' / 'tables'
PRED_DIR    = PROJECT / '04_outputs' / 'predictions'
LOG_DIR     = PROJECT / '04_outputs' / 'run_logs'
REPORT_DIR  = PROJECT / '04_outputs' / 'reports'

for d in [TABLE_DIR, PRED_DIR, LOG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT}")
print(f"Splits dir:   {SPLIT_DIR}")
print(f"Splits found: {len(list(SPLIT_DIR.glob('*.csv')))} files")

Project root: /Users/sefayet/Desktop/Github/Sarcasm_detection
Splits dir:   /Users/sefayet/Desktop/Github/Sarcasm_detection/01_data/interim/splits
Splits found: 12 files


## 1. Configuration

In [15]:
# ── Backbones to train in this notebook ──
BACKBONES = {
    'xlm-roberta':    'xlm-roberta-base',
    'mbert':          'bert-base-multilingual-cased',
    'sagorsarker-bb': 'sagorsarker/bangla-bert-base',
    'muril':          'google/muril-base-cased',
}

# ── Binary datasets ──
DATASETS = {
    'ben_sarc_binary': {
        'train': SPLIT_DIR / 'ben_sarc_binary_train.csv',
        'val':   SPLIT_DIR / 'ben_sarc_binary_val.csv',
        'test':  SPLIT_DIR / 'ben_sarc_binary_test.csv',
    },
    'banglasarc_binary': {
        'train': SPLIT_DIR / 'banglasarc_binary_train.csv',
        'val':   SPLIT_DIR / 'banglasarc_binary_val.csv',
        'test':  SPLIT_DIR / 'banglasarc_binary_test.csv',
    },
    'banglasarc3_binary': {
        'train': SPLIT_DIR / 'banglasarc3_binary_train.csv',
        'val':   SPLIT_DIR / 'banglasarc3_binary_val.csv',
        'test':  SPLIT_DIR / 'banglasarc3_binary_test.csv',
    },
}

# ── Hyperparameters (match notebook 05 for fair comparison) ──
MAX_LENGTH    = 128
EPOCHS        = 3
BATCH_SIZE    = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
SEED          = 42
METRIC_FOR_BEST = 'macro_f1'

# ── Column names (from your split CSVs) ──
TEXT_COL  = 'text'
LABEL_COL = 'label_binary'

print(f"Backbones to train: {list(BACKBONES.keys())}")
print(f"Datasets: {list(DATASETS.keys())}")
print(f"Total runs: {len(BACKBONES) * len(DATASETS)}")
print(f"Text column: '{TEXT_COL}' | Label column: '{LABEL_COL}'")

Backbones to train: ['xlm-roberta', 'mbert', 'sagorsarker-bb', 'muril']
Datasets: ['ben_sarc_binary', 'banglasarc_binary', 'banglasarc3_binary']
Total runs: 12
Text column: 'text' | Label column: 'label_binary'


## 2. Dataset Class & Metrics

In [16]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

class SarcasmDataset(Dataset):
    """Simple text-classification dataset for HF Trainer."""
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


def compute_metrics(eval_pred):
    """Compute all metrics needed for the thesis tables."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':    accuracy_score(labels, preds),
        'macro_f1':    f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
        'binary_f1':   f1_score(labels, preds, average='binary', pos_label=1),
        'precision':   precision_score(labels, preds, average='binary', pos_label=1),
        'recall':      recall_score(labels, preds, average='binary', pos_label=1),
    }


def load_split(csv_path):
    """Load a split CSV and return texts + labels."""
    df = pd.read_csv(csv_path)
    texts  = df[TEXT_COL].astype(str).tolist()
    labels = df[LABEL_COL].astype(int).tolist()
    return texts, labels


# ── Sanity check ──
for ds_name, paths in DATASETS.items():
    tr_texts, tr_labels = load_split(paths['train'])
    te_texts, te_labels = load_split(paths['test'])
    print(f"{ds_name}: train={len(tr_texts)}, test={len(te_texts)}, labels={sorted(set(tr_labels))}")

ben_sarc_binary: train=20508, test=2564, labels=[0, 1]
banglasarc_binary: train=4089, test=512, labels=[0, 1]
banglasarc3_binary: train=6413, test=802, labels=[0, 1]


## 3. Training Loop

In [17]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import time
import gc
import json
import numpy as np
import torch
from sklearn.metrics import confusion_matrix

def train_and_evaluate(
    model_tag, model_name, dataset_tag, dataset_paths,
    max_length=MAX_LENGTH, epochs=EPOCHS, batch_size=BATCH_SIZE,
    lr=LEARNING_RATE, seed=SEED
):
    """
    Train one backbone on one dataset.
    Returns a dict of results + saves predictions & checkpoint.
    """
    print(f"\n{'='*70}")
    print(f"  {model_tag} x {dataset_tag}")
    print(f"  HuggingFace model: {model_name}")
    print(f"{'='*70}")

    t_start = time.time()

    # ── Load data ──
    train_texts, train_labels = load_split(dataset_paths['train'])
    val_texts, val_labels     = load_split(dataset_paths['val'])
    test_texts, test_labels   = load_split(dataset_paths['test'])

    print(f"  Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

    # ── Tokenizer & model ──
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2
    )

    # ── Build datasets ──
    train_ds = SarcasmDataset(train_texts, train_labels, tokenizer, max_length)
    val_ds   = SarcasmDataset(val_texts,   val_labels,   tokenizer, max_length)
    test_ds  = SarcasmDataset(test_texts,  test_labels,  tokenizer, max_length)

    # ── Output directory ──
    run_name = f"07_{model_tag}_{dataset_tag}"
    output_dir = CKPT_DIR / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    # ── Training arguments ──
    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model=METRIC_FOR_BEST,
        greater_is_better=True,
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        seed=seed,
        report_to='none',
        dataloader_num_workers=2,
    )

    # ── Trainer ──
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    # ── Train ──
    trainer.train()

    # ── Evaluate on val ──
    val_results = trainer.evaluate(val_ds)
    print(f"  Val  — Acc: {val_results['eval_accuracy']:.4f} | Macro-F1: {val_results['eval_macro_f1']:.4f}")

    # ── Predict on test ──
    test_output = trainer.predict(test_ds)
    test_metrics = test_output.metrics
    test_preds = np.argmax(test_output.predictions, axis=-1)
    test_probs = torch.softmax(torch.tensor(test_output.predictions), dim=-1).numpy()
    print(f"  Test — Acc: {test_metrics['test_accuracy']:.4f} | Macro-F1: {test_metrics['test_macro_f1']:.4f}")

    # ── Predict on val (for later analysis) ──
    val_output = trainer.predict(val_ds)
    val_preds = np.argmax(val_output.predictions, axis=-1)
    val_probs = torch.softmax(torch.tensor(val_output.predictions), dim=-1).numpy()

    # ── Save predictions ──
    for split_name, texts, labels, preds, probs in [
        ('test', test_texts, test_labels, test_preds, test_probs),
        ('val',  val_texts,  val_labels,  val_preds,  val_probs),
    ]:
        pred_df = pd.DataFrame({
            'text': texts,
            'true_label': labels,
            'pred_label': preds,
            'prob_0': probs[:, 0],
            'prob_1': probs[:, 1],
        })
        pred_path = PRED_DIR / f"07_{model_tag}_{dataset_tag}_{split_name}_predictions.csv"
        pred_df.to_csv(pred_path, index=False)

    # ── Save best model (only model + tokenizer, not optimizer) ──
    best_model_dir = output_dir / 'best_model'
    trainer.save_model(str(best_model_dir))
    tokenizer.save_pretrained(str(best_model_dir))

    # ── Confusion matrix ──
    cm = confusion_matrix(test_labels, test_preds).tolist()

    t_elapsed = time.time() - t_start

    # ── Assemble result row ──
    result = {
        'model_tag':    model_tag,
        'model_name':   model_name,
        'dataset':      dataset_tag,
        'val_accuracy':    val_results['eval_accuracy'],
        'val_macro_f1':    val_results['eval_macro_f1'],
        'val_weighted_f1': val_results['eval_weighted_f1'],
        'val_binary_f1':   val_results['eval_binary_f1'],
        'test_accuracy':    test_metrics['test_accuracy'],
        'test_macro_f1':    test_metrics['test_macro_f1'],
        'test_weighted_f1': test_metrics['test_weighted_f1'],
        'test_binary_f1':   test_metrics['test_binary_f1'],
        'test_precision':   test_metrics['test_precision'],
        'test_recall':      test_metrics['test_recall'],
        'confusion_matrix': json.dumps(cm),
        'train_samples': len(train_texts),
        'epochs': epochs,
        'batch_size': batch_size,
        'lr': lr,
        'max_length': max_length,
        'time_seconds': round(t_elapsed, 1),
    }

    # ── Cleanup GPU memory ──
    del model, trainer, train_ds, val_ds, test_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## 4. Run All Experiments

4 backbones x 3 datasets = **12 runs**. On Kaggle 2xT4, expect ~3-5 hours total.

Results are saved **incrementally** after each run — if the kernel times out, just restart and it picks up where it left off.

In [18]:
import pandas as pd

all_results = []
summary_path = TABLE_DIR / '07_multi_backbone_binary_summary.csv'

# ── Resume support: load previously completed runs ──
if summary_path.exists():
    prev_df = pd.read_csv(summary_path)
    all_results = prev_df.to_dict('records')
    done_keys = set(zip(prev_df['model_tag'], prev_df['dataset']))
    print(f"Resuming: {len(done_keys)} runs already completed")
else:
    done_keys = set()

total_runs = len(BACKBONES) * len(DATASETS)
run_num = len(done_keys)

for model_tag, model_name in BACKBONES.items():
    for ds_tag, ds_paths in DATASETS.items():
        # Skip if already done
        if (model_tag, ds_tag) in done_keys:
            print(f"\nSkipping {model_tag} x {ds_tag} (already done)")
            continue

        run_num += 1
        print(f"\n>>> Run {run_num}/{total_runs}")

        try:
            result = train_and_evaluate(
                model_tag=model_tag,
                model_name=model_name,
                dataset_tag=ds_tag,
                dataset_paths=ds_paths,
            )
            all_results.append(result)

            # Save incrementally after each run
            pd.DataFrame(all_results).to_csv(summary_path, index=False)
            print(f"  Saved to {summary_path.name}")

        except Exception as e:
            print(f"  FAILED: {e}")
            # Save what we have so far
            if all_results:
                pd.DataFrame(all_results).to_csv(summary_path, index=False)
            raise

print(f"\n{'='*70}")
print(f"All {total_runs} runs complete!")
print(f"Results saved to: {summary_path}")


>>> Run 1/12

  xlm-roberta x ben_sarc_binary
  HuggingFace model: xlm-roberta-base
  Train: 20508 | Val: 2564 | Test: 2564


KeyboardInterrupt: 

## 5. Results Summary

In [ ]:
import pandas as pd
import json

# ── Load and display results ──
results_df = pd.read_csv(summary_path)

# ── Also load notebook 05 BanglaBERT baseline results for comparison ──
bb_baseline_path = TABLE_DIR / '05_baseline_banglabert_binary_summary_all_datasets.csv'
if bb_baseline_path.exists():
    bb_df = pd.read_csv(bb_baseline_path)
    print("BanglaBERT baseline results from notebook 05:")
    print(bb_df.to_string(index=False))
    print()

# ── Display new backbone results ──
display_cols = [
    'model_tag', 'dataset',
    'test_accuracy', 'test_macro_f1', 'test_binary_f1',
    'test_precision', 'test_recall', 'time_seconds'
]
print("="*90)
print("  MULTI-BACKBONE BINARY RESULTS (Test Set)")
print("="*90)
print(results_df[display_cols].to_string(index=False, float_format='%.4f'))

In [ ]:
# ── Pivot table: Macro-F1 by backbone x dataset ──
pivot = results_df.pivot_table(
    index='model_tag', columns='dataset',
    values='test_macro_f1', aggfunc='first'
)

# Add BanglaBERT from notebook 05 if available
if bb_baseline_path.exists():
    bb_df = pd.read_csv(bb_baseline_path)
    bb_row = {}
    for _, row in bb_df.iterrows():
        ds = None
        for col_name in ['dataset', 'dataset_tag', 'dataset_name']:
            if col_name in row.index:
                ds = row[col_name]
                break
        f1 = None
        for col_name in ['test_macro_f1', 'macro_f1', 'eval_macro_f1']:
            if col_name in row.index:
                f1 = row[col_name]
                break
        if ds and f1:
            bb_row[ds] = f1
    if bb_row:
        pivot.loc['banglabert (nb05)'] = bb_row

pivot['mean'] = pivot.mean(axis=1)
pivot = pivot.sort_values('mean', ascending=False)

print("="*90)
print("  MACRO-F1 COMPARISON (Test) — All Backbones x All Datasets")
print("="*90)
print(pivot.to_string(float_format='%.4f'))
print()
print(f"Best overall backbone: {pivot['mean'].idxmax()} (mean Macro-F1 = {pivot['mean'].max():.4f})")

In [ ]:
import json
import pandas as pd

# ── Save confusion matrices as JSON ──
cm_dict = {}
for _, row in results_df.iterrows():
    key = f"{row['model_tag']}_{row['dataset']}"
    cm_dict[key] = json.loads(row['confusion_matrix'])

cm_path = TABLE_DIR / '07_multi_backbone_binary_confusion_matrices.json'
with open(cm_path, 'w') as f:
    json.dump(cm_dict, f, indent=2)
print(f"Confusion matrices saved to: {cm_path.name}")

# ── Save run metadata ──
meta_cols = ['model_tag', 'model_name', 'dataset', 'train_samples',
             'epochs', 'batch_size', 'lr', 'max_length', 'time_seconds']
meta_df = results_df[meta_cols]
meta_path = LOG_DIR / '07_multi_backbone_run_metadata.csv'
meta_df.to_csv(meta_path, index=False)
print(f"Run metadata saved to: {meta_path.name}")

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

# ── Per-class classification reports ──
all_reports = []

for _, row in results_df.iterrows():
    pred_path = PRED_DIR / f"07_{row['model_tag']}_{row['dataset']}_test_predictions.csv"
    if pred_path.exists():
        pred_df = pd.read_csv(pred_path)
        report = classification_report(
            pred_df['true_label'], pred_df['pred_label'],
            output_dict=True, target_names=['Not Sarcastic', 'Sarcastic']
        )
        for cls_name, metrics in report.items():
            if isinstance(metrics, dict):
                all_reports.append({
                    'model_tag': row['model_tag'],
                    'dataset': row['dataset'],
                    'class': cls_name,
                    **metrics
                })

if all_reports:
    report_df = pd.DataFrame(all_reports)
    report_path = REPORT_DIR / '07_multi_backbone_binary_classification_reports.csv'
    report_df.to_csv(report_path, index=False)
    print(f"Classification reports saved to: {report_path.name}")

## 6. Quick Visual Check

In [ ]:
import json
import numpy as np
import pandas as pd

# ── Print confusion matrices for each run ──
for _, row in results_df.iterrows():
    cm = json.loads(row['confusion_matrix'])
    cm_arr = np.array(cm)
    print(f"\n{row['model_tag']} x {row['dataset']}")
    print(f"  Macro-F1: {row['test_macro_f1']:.4f} | Acc: {row['test_accuracy']:.4f}")
    print(f"                  Pred=0   Pred=1")
    print(f"    True=0     {cm_arr[0,0]:>6}   {cm_arr[0,1]:>6}")
    print(f"    True=1     {cm_arr[1,0]:>6}   {cm_arr[1,1]:>6}")

## 7. Wrapup

**Files produced:**
- `04_outputs/tables/07_multi_backbone_binary_summary.csv`
- `04_outputs/tables/07_multi_backbone_binary_confusion_matrices.json`
- `04_outputs/predictions/07_{model}_{dataset}_{split}_predictions.csv`
- `04_outputs/run_logs/07_multi_backbone_run_metadata.csv`
- `04_outputs/reports/07_multi_backbone_binary_classification_reports.csv`
- Model checkpoints in `03_models/checkpoints/07_{model}_{dataset}/`

**Next:** Paste the pivot table output back so I can design notebook 08.

In [ ]:
# ── Final file listing ──
print("Files produced by this notebook:")
for p in sorted(PROJECT.rglob('07_*')):
    if p.is_file():
        rel = p.relative_to(PROJECT)
        size_mb = p.stat().st_size / 1e6
        print(f"  {rel}  ({size_mb:.1f} MB)")